# YouTube Engagement Features - Virality Prediction Ensemble Models

This notebook builds ensemble models to predict viral YouTube videos using only engagement metrics (views, likes, comments, and engagement rates). All irrelevant features (audio features) are dropped to focus purely on engagement-based signals.

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
from pathlib import Path

# Preprocessing
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# Models
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve, auc
)

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')

# Set random seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("All libraries imported successfully!")

All libraries imported successfully!


## 2. Load and Explore YouTube Dataset

In [2]:
# Load dataset
DATA_PATH = "../data/processed/youtube_features_cleaned.csv"
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nColumn names:")
print(df.columns.tolist())
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nClass distribution:\n{df['viral'].value_counts()}")

Dataset shape: (31749, 87)

First few rows:
                 track_id                  track_name                 artists  \
0  5SuOikwiRyPMVoIQDJUgSV                      Comedy             Gen Hoshino   
1  4qPNDBW1i3p13qLCt0Ki3A            Ghost - Acoustic            Ben Woodward   
2  1iJBSr7s7jYXzM8EGcbK5b              To Begin Again  Ingrid Michaelson;ZAYN   
3  6lfxq3CG4xtTiEg7opyCyx  Can't Help Falling In Love            Kina Grannis   
4  5vjLSffimiIP26QG5WcN2K                     Hold On        Chord Overstreet   

   viral  virality_score  spectral_contrast_3_std  spectral_contrast_4_std  \
0      1        0.441917                 5.170993                 4.759122   
1      0        0.307049                 4.392038                 3.858416   
2      0        0.013653                 4.528816                 3.943841   
3      0        0.388813                 4.585019                 4.053476   
4      0        0.368333                 4.689249                 4.882720   



## 3. Select and Prepare Engagement Features

### 3.1 Identify Engagement Features

In [3]:
# Define engagement features (drop all audio features)
engagement_features = ['view_count', 'like_count', 'comment_count', 'like_rate', 'comment_rate']

print(f"Engagement features selected: {engagement_features}")
print(f"\nEngagement features statistics:")
print(df[engagement_features].describe())

# Check if all engagement features exist in the dataset
available_features = [f for f in engagement_features if f in df.columns]
print(f"\nAvailable engagement features: {available_features}")

# Verify no missing values in engagement features
print(f"\nMissing values in engagement features:")
print(df[available_features].isnull().sum())

Engagement features selected: ['view_count', 'like_count', 'comment_count', 'like_rate', 'comment_rate']

Engagement features statistics:
         view_count    like_count  comment_count     like_rate  comment_rate
count  3.174900e+04  3.174900e+04   3.174900e+04  31749.000000  3.174900e+04
mean   4.512710e+07  3.260602e+05   1.045607e+04      0.014178  7.627092e-04
std    2.151318e+08  1.435636e+06   5.893047e+04      0.022707  2.586327e-03
min    1.800000e+01  1.000000e+00   1.000000e+00      0.000308  2.945353e-08
25%    6.420200e+04  7.960000e+02   2.200000e+01      0.007277  1.507020e-04
50%    6.934300e+05  8.234000e+03   2.140000e+02      0.011322  3.035430e-04
75%    8.214287e+06  8.235000e+04   2.000000e+03      0.017228  6.567440e-04
max    6.965732e+09  4.638637e+07   2.400000e+06      3.553191  1.578947e-01

Available engagement features: ['view_count', 'like_count', 'comment_count', 'like_rate', 'comment_rate']

Missing values in engagement features:
view_count       0
lik

### 3.2 Create Feature and Target Dataframes

In [4]:
# Keep only engagement features and drop all irrelevant features
X = df[available_features].copy()
y = df['viral'].copy()

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeatures retained: {X.columns.tolist()}")
print(f"\nFeature statistics:")
print(X.describe())

Features shape: (31749, 5)
Target shape: (31749,)

Features retained: ['view_count', 'like_count', 'comment_count', 'like_rate', 'comment_rate']

Feature statistics:
         view_count    like_count  comment_count     like_rate  comment_rate
count  3.174900e+04  3.174900e+04   3.174900e+04  31749.000000  3.174900e+04
mean   4.512710e+07  3.260602e+05   1.045607e+04      0.014178  7.627092e-04
std    2.151318e+08  1.435636e+06   5.893047e+04      0.022707  2.586327e-03
min    1.800000e+01  1.000000e+00   1.000000e+00      0.000308  2.945353e-08
25%    6.420200e+04  7.960000e+02   2.200000e+01      0.007277  1.507020e-04
50%    6.934300e+05  8.234000e+03   2.140000e+02      0.011322  3.035430e-04
75%    8.214287e+06  8.235000e+04   2.000000e+03      0.017228  6.567440e-04
max    6.965732e+09  4.638637e+07   2.400000e+06      3.553191  1.578947e-01


## 4. Split and Scale Data

### 4.1 Stratified Train-Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"\nTraining set viral distribution:\n{y_train.value_counts()}")
print(f"\nTest set viral distribution:\n{y_test.value_counts()}")

Training set size: (25399, 5)
Test set size: (6350, 5)

Training set viral distribution:
viral
0    19049
1     6350
Name: count, dtype: int64

Test set viral distribution:
viral
0    4762
1    1588
Name: count, dtype: int64


### 4.2 Scale Engagement Features

In [6]:
# Scale all features with StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

print(f"Scaled training set shape: {X_train_scaled.shape}")
print(f"Scaled test set shape: {X_test_scaled.shape}")
print(f"\nScaled training set statistics:")
print(X_train_scaled.describe())

Scaled training set shape: (25399, 5)
Scaled test set shape: (6350, 5)

Scaled training set statistics:
         view_count    like_count  comment_count     like_rate  comment_rate
count  2.539900e+04  2.539900e+04   2.539900e+04  2.539900e+04  2.539900e+04
mean  -3.916531e-18 -2.238018e-18  -1.230910e-17  5.874797e-17 -4.224259e-17
std    1.000020e+00  1.000020e+00   1.000020e+00  1.000020e+00  1.000020e+00
min   -2.129305e-01 -2.316774e-01  -1.804200e-01 -1.254628e+00 -2.862774e-01
25%   -2.126182e-01 -2.310893e-01  -1.800306e-01 -6.163765e-01 -2.290004e-01
50%   -2.096134e-01 -2.257313e-01  -1.766327e-01 -2.498113e-01 -1.714719e-01
75%   -1.736929e-01 -1.719264e-01  -1.450430e-01  2.876993e-01 -3.937386e-02
max    3.307736e+01  3.328630e+01   4.229314e+01  1.854371e+01  5.957902e+01


## 5. Build Base Learner Models with Cross-Validation

### 5.1 XGBoost Classifier

In [7]:
# XGBClassifier with tuned parameters
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    verbosity=0,
    eval_metric='logloss'
)

# Cross-validation
cv_split = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
xgb_cv_scores = cross_val_score(xgb_model, X_train_scaled, y_train, cv=cv_split, scoring='f1_macro')

print(f"XGBoost CV F1-Macro Scores: {xgb_cv_scores}")
print(f"XGBoost Mean CV F1-Macro: {xgb_cv_scores.mean():.4f} (+/- {xgb_cv_scores.std():.4f})")

xgb_model.fit(X_train_scaled, y_train)
print("XGBoost model trained!")

XGBoost CV F1-Macro Scores: [0.9992128  0.99895068 0.99737118 0.99973746 0.99921234]
XGBoost Mean CV F1-Macro: 0.9989 (+/- 0.0008)
XGBoost model trained!


### 5.2 LightGBM Classifier

In [8]:
# LGBMClassifier with tuning
lgb_model = LGBMClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    verbose=-1
)

# Cross-validation
lgb_cv_scores = cross_val_score(lgb_model, X_train_scaled, y_train, cv=cv_split, scoring='f1_macro')

print(f"LightGBM CV F1-Macro Scores: {lgb_cv_scores}")
print(f"LightGBM Mean CV F1-Macro: {lgb_cv_scores.mean():.4f} (+/- {lgb_cv_scores.std():.4f})")

lgb_model.fit(X_train_scaled, y_train)
print("LightGBM model trained!")

LightGBM CV F1-Macro Scores: [0.99842354 0.99842685 0.99737118 0.99947507 0.99973745]
LightGBM Mean CV F1-Macro: 0.9987 (+/- 0.0008)
LightGBM model trained!


### 5.3 CatBoost Classifier

In [9]:
# CatBoostClassifier (no categorical features in engagement data)
catboost_model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    verbose=0
)

# Cross-validation
catboost_cv_scores = cross_val_score(catboost_model, X_train_scaled, y_train, cv=cv_split, scoring='f1_macro')

print(f"CatBoost CV F1-Macro Scores: {catboost_cv_scores}")
print(f"CatBoost Mean CV F1-Macro: {catboost_cv_scores.mean():.4f} (+/- {catboost_cv_scores.std():.4f})")

catboost_model.fit(X_train_scaled, y_train)
print("CatBoost model trained!")

CatBoost CV F1-Macro Scores: [0.99947507 0.99816321 0.99737118 0.99973746 0.99973758]
CatBoost Mean CV F1-Macro: 0.9989 (+/- 0.0010)
CatBoost model trained!


## 6. Train Ensemble Models

### 6.1 Soft Voting Classifier

In [10]:
# Create a voting classifier with soft voting
voting_clf = VotingClassifier(
    estimators=[
        ('xgb', XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.1, 
                             random_state=RANDOM_STATE, verbosity=0, eval_metric='logloss')),
        ('lgb', LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, 
                             random_state=RANDOM_STATE, verbose=-1)),
        ('catboost', CatBoostClassifier(iterations=300, depth=6, learning_rate=0.1, 
                                       random_state=RANDOM_STATE, verbose=0))
    ],
    voting='soft',
    n_jobs=-1
)

# Fit on full training data
voting_clf.fit(X_train_scaled, y_train)
print("Soft Voting Classifier trained!")

# Cross-validation on Voting Classifier
voting_cv_scores = cross_val_score(voting_clf, X_train_scaled, y_train, cv=cv_split, scoring='f1_macro')
print(f"Voting Classifier CV F1-Macro: {voting_cv_scores.mean():.4f} (+/- {voting_cv_scores.std():.4f})")

Soft Voting Classifier trained!
Voting Classifier CV F1-Macro: 0.9991 (+/- 0.0010)


### 6.2 Stacking Classifier

In [11]:
# Create base learners for stacking
base_learners = [
    ('xgb', XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.1, 
                         random_state=RANDOM_STATE, verbosity=0, eval_metric='logloss')),
    ('lgb', LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, 
                         random_state=RANDOM_STATE, verbose=-1)),
    ('catboost', CatBoostClassifier(iterations=300, depth=6, learning_rate=0.1, 
                                   random_state=RANDOM_STATE, verbose=0))
]

# Meta-learner
meta_learner = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1)

# Stacking classifier with passthrough=True (meta-learner sees original features too)
stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    passthrough=True,
    cv=cv_split,
    n_jobs=-1
)

# Fit stacking classifier
stacking_clf.fit(X_train_scaled, y_train)
print("Stacking Classifier trained!")

# Cross-validation on Stacking Classifier
stacking_cv_scores = cross_val_score(stacking_clf, X_train_scaled, y_train, cv=cv_split, scoring='f1_macro')
print(f"Stacking Classifier CV F1-Macro: {stacking_cv_scores.mean():.4f} (+/- {stacking_cv_scores.std():.4f})")

Stacking Classifier trained!
Stacking Classifier CV F1-Macro: 0.9992 (+/- 0.0010)


## 7. Evaluate and Compare Models

### 7.1 Evaluation Function

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    """
    Evaluate a model and return metrics.
    
    Parameters:
    - model: trained model
    - X_test: test features
    - y_test: test target
    - model_name: name of the model for display
    """
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Macro': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_pred_proba)
    }
    
    return metrics, y_pred, y_pred_proba

print("Evaluation function defined!")

### 7.2 Evaluate All Models

In [ ]:
# Evaluate all models
results = []
predictions = {}

# Evaluate base learners
metrics_xgb, pred_xgb, pred_proba_xgb = evaluate_model(xgb_model, X_test_scaled, y_test, 'XGBoost')
results.append(metrics_xgb)
predictions['XGBoost'] = (pred_xgb, pred_proba_xgb)

metrics_lgb, pred_lgb, pred_proba_lgb = evaluate_model(lgb_model, X_test_scaled, y_test, 'LightGBM')
results.append(metrics_lgb)
predictions['LightGBM'] = (pred_lgb, pred_proba_lgb)

metrics_catboost, pred_catboost, pred_proba_catboost = evaluate_model(catboost_model, X_test_scaled, y_test, 'CatBoost')
results.append(metrics_catboost)
predictions['CatBoost'] = (pred_catboost, pred_proba_catboost)

# Evaluate ensemble models
metrics_voting, pred_voting, pred_proba_voting = evaluate_model(voting_clf, X_test_scaled, y_test, 'Voting Classifier')
results.append(metrics_voting)
predictions['Voting'] = (pred_voting, pred_proba_voting)

metrics_stacking, pred_stacking, pred_proba_stacking = evaluate_model(stacking_clf, X_test_scaled, y_test, 'Stacking Classifier')
results.append(metrics_stacking)
predictions['Stacking'] = (pred_stacking, pred_proba_stacking)

# Create results dataframe
results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("MODEL EVALUATION RESULTS")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)

### 7.3 Confusion Matrices

In [ ]:
# Create confusion matrices for all models
fig, axes = plt.subplots(1, 5, figsize=(18, 3))
axes = axes.flatten()

model_names = ['XGBoost', 'LightGBM', 'CatBoost', 'Voting', 'Stacking']

for idx, (ax, model_name) in enumerate(zip(axes, model_names)):
    y_pred_model = predictions[model_name][0]
    cm = confusion_matrix(y_test, y_pred_model)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set_title(f'{model_name}')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
    ax.set_xticklabels(['Non-Viral', 'Viral'])
    ax.set_yticklabels(['Non-Viral', 'Viral'])

plt.tight_layout()
plt.savefig('../images/youtube_engagement_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()
print("Confusion matrices saved!")

### 7.4 Classification Reports

In [ ]:
# Print detailed classification reports for ensemble models
print("\n" + "="*80)
print("VOTING CLASSIFIER - Classification Report")
print("="*80)
print(classification_report(y_test, predictions['Voting'][0], 
                          target_names=['Non-Viral', 'Viral']))

print("\n" + "="*80)
print("STACKING CLASSIFIER - Classification Report")
print("="*80)
print(classification_report(y_test, predictions['Stacking'][0], 
                          target_names=['Non-Viral', 'Viral']))

### 7.5 Feature Importance Analysis

In [ ]:
# Get feature importance from XGBoost and LightGBM
xgb_importance = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

lgb_importance = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=False)

# Plot feature importances side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# XGBoost
axes[0].barh(range(len(xgb_importance)), xgb_importance['importance'].values, color='skyblue')
axes[0].set_yticks(range(len(xgb_importance)))
axes[0].set_yticklabels(xgb_importance['feature'].values)
axes[0].set_xlabel('Importance')
axes[0].set_title('XGBoost - Feature Importances')
axes[0].grid(axis='x', alpha=0.3)

# LightGBM
axes[1].barh(range(len(lgb_importance)), lgb_importance['importance'].values, color='lightcoral')
axes[1].set_yticks(range(len(lgb_importance)))
axes[1].set_yticklabels(lgb_importance['feature'].values)
axes[1].set_xlabel('Importance')
axes[1].set_title('LightGBM - Feature Importances')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../images/youtube_engagement_feature_importances.png', dpi=300, bbox_inches='tight')
plt.show()
print("Feature importance plot saved!")

print("\nXGBoost Feature Importances:")
print(xgb_importance)
print("\nLightGBM Feature Importances:")
print(lgb_importance)

### 7.6 ROC Curves Comparison

In [ ]:
# Plot ROC curves for all models
fig, ax = plt.subplots(figsize=(12, 8))

models_to_plot = ['XGBoost', 'LightGBM', 'CatBoost', 'Voting', 'Stacking']
colors = plt.cm.tab10(np.linspace(0, 1, len(models_to_plot)))

for model_name, color in zip(models_to_plot, colors):
    y_pred_proba = predictions[model_name][1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{model_name} (AUC = {roc_auc:.3f})', color=color, linewidth=2)

# Plot diagonal
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves - YouTube Engagement Models Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../images/youtube_engagement_roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print("ROC curves saved!")

## 8. Save Best Model and Artifacts

### 8.1 Identify Best Model

In [ ]:
# Identify best model by F1-Macro score
best_model_idx = results_df['F1-Macro'].idxmax()
best_model_name = results_df.loc[best_model_idx, 'Model']
best_f1_score = results_df.loc[best_model_idx, 'F1-Macro']

print(f"\nBest Model: {best_model_name}")
print(f"F1-Macro Score: {best_f1_score:.4f}")
print(f"\nBest Model Metrics:")
print(results_df.loc[best_model_idx].to_string())

### 8.2 Save All Models and Artifacts

In [ ]:
# Select the best model object
best_model_mapping = {
    'XGBoost': xgb_model,
    'LightGBM': lgb_model,
    'CatBoost': catboost_model,
    'Voting Classifier': voting_clf,
    'Stacking Classifier': stacking_clf
}

best_model = best_model_mapping[best_model_name]

# Create models directory if it doesn't exist
models_dir = Path('../models')
models_dir.mkdir(parents=True, exist_ok=True)

# Save best model
model_save_path = models_dir / 'youtube_engagement_best_model.pkl'
joblib.dump(best_model, model_save_path)
print(f"Best model saved to: {model_save_path}")
print(f"File size: {model_save_path.stat().st_size / 1024:.2f} KB")

# Save all individual models
all_models = {
    'youtube_engagement_xgb': xgb_model,
    'youtube_engagement_lgb': lgb_model,
    'youtube_engagement_catboost': catboost_model,
    'youtube_engagement_voting': voting_clf,
    'youtube_engagement_stacking': stacking_clf
}

for name, model in all_models.items():
    model_path = models_dir / f'{name}.pkl'
    joblib.dump(model, model_path)
    print(f"Saved: {name}")

# Save scaler
scaler_path = models_dir / 'youtube_engagement_scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f"Saved: youtube_engagement_scaler")

# Save results dataframe
results_path = models_dir / 'youtube_engagement_evaluation_results.csv'
results_df.to_csv(results_path, index=False)
print(f"Saved: youtube_engagement_evaluation_results")

## 9. Summary and Conclusions

In [ ]:
print("\n" + "="*80)
print("YOUTUBE ENGAGEMENT FEATURES - VIRALITY PREDICTION SUMMARY")
print("="*80)
print(f"\nDataset Information:")
print(f"  - Total samples: {len(df)}")
print(f"  - Training samples: {len(X_train)}")
print(f"  - Test samples: {len(X_test)}")
print(f"  - Number of engagement features: {X_train_scaled.shape[1]}")
print(f"  - Features used: {', '.join(X_train_scaled.columns.tolist())}")

print(f"\nClass Distribution:")
print(f"  - Training: {dict(y_train.value_counts())}")
print(f"  - Test: {dict(y_test.value_counts())}")

print(f"\nModels Trained:")
print(f"  1. XGBoost (base learner)")
print(f"  2. LightGBM (base learner)")
print(f"  3. CatBoost (base learner)")
print(f"  4. Voting Classifier (soft voting ensemble)")
print(f"  5. Stacking Classifier (meta-learner ensemble)")

print(f"\nBest Performing Model: {best_model_name}")
print(f"  - F1-Macro Score: {results_df.loc[best_model_idx, 'F1-Macro']:.4f}")
print(f"  - ROC-AUC Score: {results_df.loc[best_model_idx, 'ROC-AUC']:.4f}")
print(f"  - Accuracy: {results_df.loc[best_model_idx, 'Accuracy']:.4f}")
print(f"  - Precision: {results_df.loc[best_model_idx, 'Precision']:.4f}")
print(f"  - Recall: {results_df.loc[best_model_idx, 'Recall']:.4f}")

print(f"\nAll Model Performance:")
print(results_df.to_string(index=False))

print(f"\nArtifacts Saved:")
print(f"  - Best model: youtube_engagement_best_model.pkl")
print(f"  - Individual models: youtube_engagement_[xgb|lgb|catboost|voting|stacking].pkl")
print(f"  - Scaler: youtube_engagement_scaler.pkl")
print(f"  - Results CSV: youtube_engagement_evaluation_results.csv")
print(f"  - Visualizations (notebooks/images/): youtube_engagement_[confusion_matrices|feature_importances|roc_curves].png")

print(f"\nKey Insights:")
print(f"  - Models trained on {X_train_scaled.shape[1]} engagement features only")
print(f"  - All audio features were excluded to focus on engagement signals")
print(f"  - Engagement features: view_count, like_count, comment_count, like_rate, comment_rate")

print("\n" + "="*80)